In [35]:
SQLITE_2_PATH = './data/db3.sqlite3'
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [3]:
def execute_query(db, query):
    conn = sqlite3.connect(db)
    cur = conn.cursor()
    cur.execute(query)
    rows = cur.fetchall()
    conn.commit()
    conn.close()
    return rows

In [4]:
data = pd.DataFrame(execute_query(SQLITE_2_PATH, "SELECT * FROM moves;"))

In [5]:
data

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,Blitz,1728,0,e2e4,0.1,0.10,-0.02,0,1,0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,0,20
1,Blitz,1728,2,g1f3,0.12,0.12,0.03,0,1,0,rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBN...,0,30
2,Blitz,1728,4,d2d4,0.48,0.48,0.21,0,1,0,rnbqkbnr/1ppp1ppp/p3p3/8/4P3/5N2/PPPP1PPP/RNBQ...,0,28
3,Blitz,1728,6,b2b3,0.66,0.66,-0.31,0,1,0,rnbqkbnr/2pp1ppp/p3p3/1p6/3PP3/5N2/PPP2PPP/RNB...,0,35
4,Blitz,1728,8,f1d3,0.82,0.82,0.10,0,1,0,rn1qkbnr/1bpp1ppp/p3p3/1p6/3PP3/1P3N2/P1P2PPP/...,0,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...
34457,Blitz,1792,68,f4e5,-1.36,-1.36,0.15,0,1,1,2r3k1/1pPq1p1p/p5p1/8/5Q2/P7/6PK/2R5 w - - 2 35,0,38
34458,Blitz,1792,70,c1c6,61.52,61.52,0.10,0,1,1,2r3k1/1pP2p1p/p1q3p1/4Q3/8/P7/6PK/2R5 w - - 4 36,0,43
34459,Blitz,1792,72,e5d6,61.37,61.37,0.00,0,1,1,2r3k1/2P2p1p/p1p3p1/4Q3/8/P7/6PK/8 w - - 0 37,0,31
34460,Blitz,1792,74,d6d8,14.15,14.15,0.33,0,1,1,2r5/2P2pkp/p1pQ2p1/8/8/P7/6PK/8 w - - 2 38,0,25


In [6]:
data.rename(columns={0: 'type', 1:'white_elo', 2:'move_ply', 3:'move', 4:'cp', 5:'cp_rel', 6:'cp_loss', 7:'is_blunder_cp', 8:'white_active', 9:'low_time', 10:'board', 11:'is_check', 12:'num_legal_moves'}, inplace=True)

In [7]:
data

,type,white_elo,move_ply,move,cp,cp_rel,cp_loss,is_blunder_cp,white_active,low_time,board,is_check,num_legal_moves
0,Blitz,1728,0,e2e4,0.1,0.10,-0.02,0,1,0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,0,20
1,Blitz,1728,2,g1f3,0.12,0.12,0.03,0,1,0,rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBN...,0,30
2,Blitz,1728,4,d2d4,0.48,0.48,0.21,0,1,0,rnbqkbnr/1ppp1ppp/p3p3/8/4P3/5N2/PPPP1PPP/RNBQ...,0,28
3,Blitz,1728,6,b2b3,0.66,0.66,-0.31,0,1,0,rnbqkbnr/2pp1ppp/p3p3/1p6/3PP3/5N2/PPP2PPP/RNB...,0,35
4,Blitz,1728,8,f1d3,0.82,0.82,0.10,0,1,0,rn1qkbnr/1bpp1ppp/p3p3/1p6/3PP3/1P3N2/P1P2PPP/...,0,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...
34457,Blitz,1792,68,f4e5,-1.36,-1.36,0.15,0,1,1,2r3k1/1pPq1p1p/p5p1/8/5Q2/P7/6PK/2R5 w - - 2 35,0,38
34458,Blitz,1792,70,c1c6,61.52,61.52,0.10,0,1,1,2r3k1/1pP2p1p/p1q3p1/4Q3/8/P7/6PK/2R5 w - - 4 36,0,43
34459,Blitz,1792,72,e5d6,61.37,61.37,0.00,0,1,1,2r3k1/2P2p1p/p1p3p1/4Q3/8/P7/6PK/8 w - - 0 37,0,31
34460,Blitz,1792,74,d6d8,14.15,14.15,0.33,0,1,1,2r5/2P2pkp/p1pQ2p1/8/8/P7/6PK/8 w - - 2 38,0,25


In [8]:
x = data['board']
y = data['move']

In [9]:
one_board = x[0]
one_board

'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1'

Esta es la info de un board. Tiene la siguiente información: 
1. Disposición de las piezas en el tablero. Negras son minúsculas y Blancas son mayúsculas.
2. w o b para saber el turno. Ya tenemos esa info en white active (1 o 0).
3. KQ kq es si tiene derecho a enrocarse las blancas o las negras en dirección del rey o la reina.
4. Guion que no dice nada.
5. Quien es activo en este par de jugadas (par Blancas, impar Negras). Se subsime en 2.
6. Ply. Me da igual, lo tengo en move_ply.

Por tanto, vamos a borrar la info que no me interesa y separar la info que sí en una nueva columna.

In [10]:
splitted = one_board.split(sep=' ')
splitted

['rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR', 'w', 'KQkq', '-', '0', '1']

In [11]:
list_splitted = [xi.split(sep=' ') for xi in x]
castle_rights = [row[2] for row in list_splitted]
castle_rights

['KQkq',
 'KQkq',
 'KQkq',
 'KQkq',
 'KQkq',
 'KQkq',
 'KQkq',
 'kq',
 'kq',
 'kq',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 'KQkq',
 'KQkq',
 'KQkq',
 'KQkq',
 'KQkq',
 'KQkq',
 'KQkq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'kq',
 'k',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 'KQkq',
 'KQkq',
 'KQkq',
 'KQkq',
 'KQkq',
 'KQkq',
 'KQ',
 'KQ',
 'KQ',
 'KQ',
 'KQ',
 'KQ',
 'KQ',
 'KQ',
 'KQ',
 'KQ',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 '-',
 'KQkq'

In [12]:
def get_rights_column(right, data):
    rights = []
    for r in data:
        if right in r:
            rights.append(1)
        else:
            rights.append(0)
    return rights


In [13]:
def add_castle_rights(data, rights):
    data['white_king_rights'] = get_rights_column('K', rights)
    data['white_queen_rights'] = get_rights_column('Q', rights)
    data['black_king_rights'] = get_rights_column('k', rights)
    data['black_queen_rights'] = get_rights_column('q', rights)
    return data

In [14]:
add_castle_rights(data, castle_rights)

,type,white_elo,move_ply,move,cp,cp_rel,cp_loss,is_blunder_cp,white_active,low_time,board,is_check,num_legal_moves,white_king_rights,white_queen_rights,black_king_rights,black_queen_rights
0,Blitz,1728,0,e2e4,0.1,0.10,-0.02,0,1,0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,0,20,1,1,1,1
1,Blitz,1728,2,g1f3,0.12,0.12,0.03,0,1,0,rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBN...,0,30,1,1,1,1
2,Blitz,1728,4,d2d4,0.48,0.48,0.21,0,1,0,rnbqkbnr/1ppp1ppp/p3p3/8/4P3/5N2/PPPP1PPP/RNBQ...,0,28,1,1,1,1
3,Blitz,1728,6,b2b3,0.66,0.66,-0.31,0,1,0,rnbqkbnr/2pp1ppp/p3p3/1p6/3PP3/5N2/PPP2PPP/RNB...,0,35,1,1,1,1
4,Blitz,1728,8,f1d3,0.82,0.82,0.10,0,1,0,rn1qkbnr/1bpp1ppp/p3p3/1p6/3PP3/1P3N2/P1P2PPP/...,0,36,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34457,Blitz,1792,68,f4e5,-1.36,-1.36,0.15,0,1,1,2r3k1/1pPq1p1p/p5p1/8/5Q2/P7/6PK/2R5 w - - 2 35,0,38,0,0,0,0
34458,Blitz,1792,70,c1c6,61.52,61.52,0.10,0,1,1,2r3k1/1pP2p1p/p1q3p1/4Q3/8/P7/6PK/2R5 w - - 4 36,0,43,0,0,0,0
34459,Blitz,1792,72,e5d6,61.37,61.37,0.00,0,1,1,2r3k1/2P2p1p/p1p3p1/4Q3/8/P7/6PK/8 w - - 0 37,0,31,0,0,0,0
34460,Blitz,1792,74,d6d8,14.15,14.15,0.33,0,1,1,2r5/2P2pkp/p1pQ2p1/8/8/P7/6PK/8 w - - 2 38,0,25,0,0,0,0


In [15]:
def leave_board_only(data):
    data['board'] = [board.split(sep=' ')[0] for board in data['board']]
    return data

In [16]:
leave_board_only(data)

,type,white_elo,move_ply,move,cp,cp_rel,cp_loss,is_blunder_cp,white_active,low_time,board,is_check,num_legal_moves,white_king_rights,white_queen_rights,black_king_rights,black_queen_rights
0,Blitz,1728,0,e2e4,0.1,0.10,-0.02,0,1,0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR,0,20,1,1,1,1
1,Blitz,1728,2,g1f3,0.12,0.12,0.03,0,1,0,rnbqkbnr/pppp1ppp/4p3/8/4P3/8/PPPP1PPP/RNBQKBNR,0,30,1,1,1,1
2,Blitz,1728,4,d2d4,0.48,0.48,0.21,0,1,0,rnbqkbnr/1ppp1ppp/p3p3/8/4P3/5N2/PPPP1PPP/RNBQ...,0,28,1,1,1,1
3,Blitz,1728,6,b2b3,0.66,0.66,-0.31,0,1,0,rnbqkbnr/2pp1ppp/p3p3/1p6/3PP3/5N2/PPP2PPP/RNB...,0,35,1,1,1,1
4,Blitz,1728,8,f1d3,0.82,0.82,0.10,0,1,0,rn1qkbnr/1bpp1ppp/p3p3/1p6/3PP3/1P3N2/P1P2PPP/...,0,36,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34457,Blitz,1792,68,f4e5,-1.36,-1.36,0.15,0,1,1,2r3k1/1pPq1p1p/p5p1/8/5Q2/P7/6PK/2R5,0,38,0,0,0,0
34458,Blitz,1792,70,c1c6,61.52,61.52,0.10,0,1,1,2r3k1/1pP2p1p/p1q3p1/4Q3/8/P7/6PK/2R5,0,43,0,0,0,0
34459,Blitz,1792,72,e5d6,61.37,61.37,0.00,0,1,1,2r3k1/2P2p1p/p1p3p1/4Q3/8/P7/6PK/8,0,31,0,0,0,0
34460,Blitz,1792,74,d6d8,14.15,14.15,0.33,0,1,1,2r5/2P2pkp/p1pQ2p1/8/8/P7/6PK/8,0,25,0,0,0,0


Ya sólo queda tratar los boards para que sea utilizable por la IA

In [17]:
data['board'][0]

'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR'

In [18]:
# piece_to_one_hot = {
#     'r':[1,0,0,0,0,0,0,0,0,0,0,0],
#     'n':[0,1,0,0,0,0,0,0,0,0,0,0],
#     'b':[0,0,1,0,0,0,0,0,0,0,0,0],
#     'q':[0,0,0,1,0,0,0,0,0,0,0,0],
#     'k':[0,0,0,0,1,0,0,0,0,0,0,0],
#     'p':[0,0,0,0,0,1,0,0,0,0,0,0],
#     'R':[0,0,0,0,0,0,1,0,0,0,0,0],
#     'N':[0,0,0,0,0,0,0,1,0,0,0,0],
#     'B':[0,0,0,0,0,0,0,0,1,0,0,0],
#     'Q':[0,0,0,0,0,0,0,0,0,1,0,0],
#     'K':[0,0,0,0,0,0,0,0,0,0,1,0],
#     'P':[0,0,0,0,0,0,0,0,0,0,0,1]
# }
#empty_square = [0,0,0,0,0,0,0,0,0,0,0,0]

In [19]:
piece_to_one_hot = {
    'r':[-1.,0.,0.,0.,0.,0.],
    'n':[0.,-1.,0.,0.,0.,0.],
    'b':[0.,0.,-1.,0.,0.,0.],
    'q':[0.,0.,0.,-1.,0.,0.],
    'k':[0.,0.,0.,0.,-1.,0.],
    'p':[0.,0.,0.,0.,0.,-1.],
    'R':[1.,0.,0.,0.,0.,0.],
    'N':[0.,1.,0.,0.,0.,0.],
    'B':[0.,0.,1.,0.,0.,0.],
    'Q':[0.,0.,0.,1.,0.,0.],
    'K':[0.,0.,0.,0.,1.,0.],
    'P':[0.,0.,0.,0.,0.,1.]
}
empty_square = [0.,0.,0.,0.,0.,0.]

In [20]:
def get_one_hot(board: str):
    one_hot_board = []
    rows = board.split(sep='/')
    for row in rows:
        one_hot_row = []
        for piece in [*row]:
            if piece.isdigit():
                for i in range(int(piece)):
                    one_hot_row.append(empty_square)
            else:
                one_hot_row.append(piece_to_one_hot[piece])
        one_hot_board.append(one_hot_row)
    return one_hot_board

In [21]:
one_hot_board = get_one_hot(data['board'][0])

In [22]:
one_hot_board[0]

[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, -1.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, -1.0, 0.0],
 [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
 [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
 [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0]]

Ahora se lo aplico a toda la columna de pd

In [23]:
data['board'] = data['board'].apply(get_one_hot)

In [24]:
data['board'][0]

[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, -1.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, -1.0, 0.0],
  [0.0, 0.0, -1.0, 0.0, 0.0, 0.0],
  [0.0, -1.0, 0.0, 0.0, 0.0, 0.0],
  [-1.0, 0.0, 0.0, 0.0, 0.0, 0.0]],
 [[0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, -1.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, -1.0]],
 [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]],
 [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 0

In [25]:
data

,type,white_elo,move_ply,move,cp,cp_rel,cp_loss,is_blunder_cp,white_active,low_time,board,is_check,num_legal_moves,white_king_rights,white_queen_rights,black_king_rights,black_queen_rights
0,Blitz,1728,0,e2e4,0.1,0.10,-0.02,0,1,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,20,1,1,1,1
1,Blitz,1728,2,g1f3,0.12,0.12,0.03,0,1,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,30,1,1,1,1
2,Blitz,1728,4,d2d4,0.48,0.48,0.21,0,1,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,28,1,1,1,1
3,Blitz,1728,6,b2b3,0.66,0.66,-0.31,0,1,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,35,1,1,1,1
4,Blitz,1728,8,f1d3,0.82,0.82,0.10,0,1,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,36,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34457,Blitz,1792,68,f4e5,-1.36,-1.36,0.15,0,1,1,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,38,0,0,0,0
34458,Blitz,1792,70,c1c6,61.52,61.52,0.10,0,1,1,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,43,0,0,0,0
34459,Blitz,1792,72,e5d6,61.37,61.37,0.00,0,1,1,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,31,0,0,0,0
34460,Blitz,1792,74,d6d8,14.15,14.15,0.33,0,1,1,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,25,0,0,0,0


Dividimos el movimiento en 2, uno de inicial y otro de final

In [26]:
data['initial_pos'] = [move[:2] for move in data['move']]
data['final_pos'] = [move[2:4] for move in data['move']]

In [27]:
data

,type,white_elo,move_ply,move,cp,cp_rel,cp_loss,is_blunder_cp,white_active,low_time,board,is_check,num_legal_moves,white_king_rights,white_queen_rights,black_king_rights,black_queen_rights,initial_pos,final_pos
0,Blitz,1728,0,e2e4,0.1,0.10,-0.02,0,1,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,20,1,1,1,1,e2,e4
1,Blitz,1728,2,g1f3,0.12,0.12,0.03,0,1,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,30,1,1,1,1,g1,f3
2,Blitz,1728,4,d2d4,0.48,0.48,0.21,0,1,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,28,1,1,1,1,d2,d4
3,Blitz,1728,6,b2b3,0.66,0.66,-0.31,0,1,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,35,1,1,1,1,b2,b3
4,Blitz,1728,8,f1d3,0.82,0.82,0.10,0,1,0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",0,36,1,1,1,1,f1,d3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34457,Blitz,1792,68,f4e5,-1.36,-1.36,0.15,0,1,1,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,38,0,0,0,0,f4,e5
34458,Blitz,1792,70,c1c6,61.52,61.52,0.10,0,1,1,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,43,0,0,0,0,c1,c6
34459,Blitz,1792,72,e5d6,61.37,61.37,0.00,0,1,1,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,31,0,0,0,0,e5,d6
34460,Blitz,1792,74,d6d8,14.15,14.15,0.33,0,1,1,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,25,0,0,0,0,d6,d8


In [28]:
final_data = data[['board', 'white_king_rights', 'white_queen_rights', 'black_king_rights', 'black_queen_rights', 'initial_pos', 'final_pos']]

In [29]:
final_data

,board,white_king_rights,white_queen_rights,black_king_rights,black_queen_rights,initial_pos,final_pos
0,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",1,1,1,1,e2,e4
1,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",1,1,1,1,g1,f3
2,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",1,1,1,1,d2,d4
3,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",1,1,1,1,b2,b3
4,"[[[-1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, -1.0,...",1,1,1,1,f1,d3
...,...,...,...,...,...,...,...
34457,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,0,0,0,f4,e5
34458,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,0,0,0,c1,c6
34459,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,0,0,0,e5,d6
34460,"[[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0...",0,0,0,0,d6,d8


In [30]:
type(final_data['board'][0][0][0][0])

float

In [32]:
train_val_df, test_df = train_test_split(final_data, test_size=0.1, random_state=47, stratify=final_data['initial_pos'])
train_df, val_df = train_test_split(train_val_df, test_size=0.1, random_state=47, stratify=train_val_df['initial_pos'])

In [33]:
print("Train: {}".format(len(train_df)))
print("Validation: {}".format(len(val_df)))
print("Test: {}".format(len(test_df)))

Train: 27913
Validation: 3102
Test: 3447


In [34]:
train_df.to_csv("./data/ml_data_train.csv")
val_df.to_csv("./data/ml_data_val.csv")
test_df.to_csv("./data/ml_data_test.csv")